## This Notebook will be divided into Three parts:
### 1) Feature Selection
### 2) Train test split and Encoding
### 3) Model Selection
### 4) Hyperparameter Tuning
### 5) Saving the model

#### ------------------------------------------------------------------------------------------------------------------------------------------

In [44]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

## 1) Feature Selection & Feature Engineering

In [45]:
df=pd.read_csv('train.csv')

In [46]:
features_with_na=[features for features in df.columns if df[features].isnull().sum()>=1]
for feature in features_with_na:
    print(feature,df[feature].isnull().sum(), "missing value")

LotFrontage 259 missing value
Alley 1369 missing value
MasVnrType 872 missing value
MasVnrArea 8 missing value
BsmtQual 37 missing value
BsmtCond 37 missing value
BsmtExposure 38 missing value
BsmtFinType1 37 missing value
BsmtFinType2 38 missing value
Electrical 1 missing value
FireplaceQu 690 missing value
GarageType 81 missing value
GarageYrBlt 81 missing value
GarageFinish 81 missing value
GarageQual 81 missing value
GarageCond 81 missing value
PoolQC 1453 missing value
Fence 1179 missing value
MiscFeature 1406 missing value


In [47]:
df['LotFrontage']=df['LotFrontage'].fillna(df['LotFrontage'].median())

In [48]:
df['Alley']=df['Alley'].fillna('NA')
df['MasVnrArea']=df['MasVnrArea'].fillna(0)

In [49]:
df.loc[df['MasVnrArea'] == 0, 'MasVnrType'] = 'None'
df['MasVnrType'] = df['MasVnrType'].fillna(df['MasVnrType'].mode()[0])

In [50]:
df['BsmtQual'] = df['BsmtQual'].fillna('NA')

mask = df['BsmtQual'] == 'NA'

df.loc[mask, 'BsmtCond'] = 'NA'
df.loc[mask, 'BsmtExposure'] = 'NA'
df.loc[mask, 'BsmtFinType1'] = 'NA'
df.loc[mask, 'BsmtFinType2'] = 'NA'

In [51]:
df['BsmtExposure'] = df['BsmtExposure'].fillna(df['BsmtExposure'].mode()[0])
df['BsmtFinType2'] = df['BsmtFinType2'].fillna(df['BsmtFinType2'].mode()[0])
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [52]:
df.loc[df['Fireplaces']==0, 'FireplaceQu']= 'NA'

In [53]:
df['GarageAge'] = df['YrSold'] - df['GarageYrBlt']

In [54]:
df['GarageType']=df['GarageType'].fillna('NA')

mask = df['GarageType'] == 'NA'

df.loc[mask, 'GarageAge'] = 0
df.loc[mask, 'GarageFinish'] = 'NA'
df.loc[mask, 'GarageQual'] = 'NA'
df.loc[mask, 'GarageCond'] = 'NA'

In [55]:
df=df.drop('GarageYrBlt',axis=1)

In [56]:
df['PoolQC']=df['PoolQC'].fillna('NA')
df['Fence']=df['Fence'].fillna('NA')
df['MiscFeature']=df['MiscFeature'].fillna('NA')

In [57]:
X=df.drop('SalePrice',axis=1)
y=df['SalePrice']

In [58]:
X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
X['Total_sqr_footage'] = X['BsmtFinSF1'] + X['BsmtFinSF2'] + X['1stFlrSF'] + X['2ndFlrSF']
X['TotalBathrooms'] = X['FullBath']+ 0.5 * X['HalfBath']+ X['BsmtFullBath']+ 0.5 * X['BsmtHalfBath']
X['TotalPorchSF'] = X['OpenPorchSF']+ X['3SsnPorch']+ X['EnclosedPorch']+ X['ScreenPorch']+ X['WoodDeckSF']

In [ ]:
# X=X.drop(['TotalBsmtSF','1stFlrSF','2ndFlrSF','BsmtFinSF1','BsmtFinSF2','FullBath','HalfBath','BsmtFullBath','BsmtHalfBath',
#           'OpenPorchSF','3SsnPorch','EnclosedPorch','ScreenPorch','WoodDeckSF'],axis=1)

In [60]:
X['PropertyAge']=X['YrSold']-X['YearRemodAdd']
X=X.drop(['YearBuilt','YearRemodAdd',],axis=1)

In [61]:
X['MSSubClass']=X['MSSubClass'].astype('str')

In [62]:
numerical_features=[features for features in X.columns if X[features].dtype!='str']
print(len(numerical_features))

cat_features=[features for features in X.columns if X[features].dtype=='str']
print(len(cat_features))

OrdinalFeatures=['ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','FireplaceQu',
                 'GarageQual','GarageCond','GarageFinish','PoolQC','Fence']

cat_features = [item for item in cat_features if item not in OrdinalFeatures]
print(len(cat_features))
print(OrdinalFeatures)

25
44
29
['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'GarageFinish', 'PoolQC', 'Fence']


In [63]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [64]:
# X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.10,random_state=42)

In [65]:
preprocessor=ColumnTransformer(
    transformers=[
        (
            'OHE',
            OneHotEncoder(handle_unknown='ignore',drop='first'),
            cat_features
        ),
        (
            'OE',
            OrdinalEncoder(categories=[
                ['Po','Fa','TA','Gd','Ex'],
                ['Po','Fa','TA','Gd','Ex'],
                ['NA','Po','Fa','TA','Gd','Ex'],
                ['NA','Po','Fa','TA','Gd','Ex'],
                ['NA','No','Mn','Av','Gd'],
                ['NA','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
                ['NA','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
                ['Po','Fa','TA','Gd','Ex'],
                ['Po','Fa','TA','Gd','Ex'],
                ['NA','Po','Fa','TA','Gd','Ex'],
                ['NA','Po','Fa','TA','Gd','Ex'],
                ['NA','Po','Fa','TA','Gd','Ex'],
                ['NA','Unf','RFn','Fin'],
                ['NA','Fa','TA','Gd','Ex'],
                ['NA','MnWw','GdWo','MnPrv','GdPrv']
            ]),
            OrdinalFeatures
        ),
        (
            'Scaler',
            StandardScaler(),
            numerical_features
        )
    ]
)

In [66]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [67]:
# ## Create a Function to Evaluate Model

# def evaluate_model(true, predicted):
#     mae = mean_absolute_error(true, predicted)
#     mse = mean_squared_error(true, predicted)
#     rmse = np.sqrt(mean_squared_error(true, predicted))
#     rmsle = np.sqrt(
#         np.mean(
#             (np.log1p(true) - np.log1p(predicted)) ** 2
#         )
#     )
#     r2_square = r2_score(true, predicted)
#     return mae, rmse, rmsle, r2_square


In [68]:
# models = {
#     "Linear Regression": LinearRegression(),
#     "Lasso": Lasso(),
#     "Ridge": Ridge(),
#     "K-Neighbors Regressor": KNeighborsRegressor(),
#     "Decision Tree": DecisionTreeRegressor(),
#     "Random Forest Regressor": RandomForestRegressor(),
#     "Adaboost Regressor": AdaBoostRegressor(),
#     "Graident BoostRegressor": GradientBoostingRegressor(),
#     "Xgboost Regressor": XGBRegressor(),
#     "LightGBM Regressor": LGBMRegressor(),
#     "CatBoost Regressor": CatBoostRegressor()
   
# }

# for i in range(len(list(models))):

#     model = list(models.values())[i]

#     pipeline = Pipeline(
#         steps=[
#             ('preprocessor', preprocessor),
#             ('model', model)
#         ]
#     )

#     pipeline.fit(X_train, y_train)  # Train model

#     # Make predictions
#     y_train_pred = pipeline.predict(X_train)
#     y_test_pred = pipeline.predict(X_test)
    
#     # Evaluate Train and Test dataset
#     model_train_mae, model_train_rmse, model_train_rmsle, model_train_r2 = evaluate_model(
#         y_train, y_train_pred
#     )

#     model_test_mae, model_test_rmse, model_test_rmsle, model_test_r2 = evaluate_model(
#         y_test, y_test_pred
#     )

    
#     print(list(models.keys())[i])
    
#     print('Model performance for Training set')
#     print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
#     print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
#     print("- RMSLE: {:.4f}".format(model_train_rmsle))
#     print("- R2 Score: {:.4f}".format(model_train_r2))

#     print('----------------------------------')
    
#     print('Model performance for Test set')
#     print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
#     print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
#     print("- RMSLE: {:.4f}".format(model_test_rmsle))
#     print("- R2 Score: {:.4f}".format(model_test_r2))
    
#     print('=' * 35)
#     print('\n')

In [69]:
# #Initialize few parameter for Hyperparamter tuning

# rf_params = {
#     'n_estimators': [200, 300, 500],
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': [1.0, 'sqrt', 'log2']
# }
# gb_params = {
#     'n_estimators': [100, 200, 300, 500],
#     'learning_rate': [0.01, 0.03, 0.05, 0.1],
#     'max_depth': [2, 3, 4, 5],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'subsample': [0.7, 0.8, 0.9, 1.0]
# }
# xgb_params = {
#     'n_estimators': [200, 500, 1000],
#     'learning_rate': [0.01, 0.03, 0.05, 0.1],
#     'max_depth': [3, 4, 5, 6],
#     'min_child_weight': [1, 3, 5],
#     'subsample': [0.7, 0.8, 0.9, 1.0],
#     'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
#     'gamma': [0, 0.1, 0.5, 1],
#     'reg_alpha': [0, 0.01, 0.1],
#     'reg_lambda': [1, 5, 10]
# }
# lgb_params = {
#     'n_estimators': [200, 500, 1000],
#     'learning_rate': [0.01, 0.03, 0.05, 0.1],
#     'num_leaves': [15, 31, 50, 70, 100],
#     'max_depth': [-1, 5, 10, 15],
#     'min_child_samples': [10, 20, 30, 50],
#     'subsample': [0.7, 0.8, 0.9, 1.0],
#     'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
#     'reg_alpha': [0, 0.01, 0.1, 1],
#     'reg_lambda': [0.1, 1, 5, 10]
# }
# cat_params = {
#     'iterations': [300, 500, 1000, 1500],
#     'learning_rate': [0.01, 0.03, 0.05, 0.1],
#     'depth': [4, 5, 6, 7, 8, 10],
#     'l2_leaf_reg': [1, 3, 5, 7, 10],
#     'random_strength': [0, 0.5, 1, 2],
#     'bagging_temperature': [0, 0.5, 1, 2]
# }

In [70]:
# randomcv_models = [
#                    ("RF", RandomForestRegressor(), rf_params),
#                    ("GradientBoost",GradientBoostingRegressor(),gb_params),
#                    ("XGboost",XGBRegressor(),xgb_params),
#                    ("LightGBM",LGBMRegressor(),lgb_params),
#                    ("CatBoost",CatBoostRegressor(),cat_params)
                   
#                    ]

In [71]:
# ##Hyperparameter Tuning
# X_train_processed=preprocessor.fit_transform(X_train)
# from sklearn.model_selection import RandomizedSearchCV

# random = RandomizedSearchCV(estimator=RandomForestRegressor(), param_distributions=rf_params,n_iter=100, scoring='neg_root_mean_squared_log_error', verbose=2,
#                              random_state=42,  n_jobs=-1)
# random.fit(X_train_processed, y_train)

In [72]:
# random = RandomizedSearchCV(estimator=GradientBoostingRegressor(), param_distributions=gb_params,n_iter=100, scoring='neg_root_mean_squared_log_error', verbose=2,
#                              random_state=42,  n_jobs=-1)
# random.fit(X_train_processed, y_train)

In [73]:
# random = RandomizedSearchCV(estimator=XGBRegressor(), param_distributions=xgb_params,n_iter=100, scoring='neg_root_mean_squared_log_error', verbose=2,
#                              random_state=42,  n_jobs=-1)
# random.fit(X_train_processed, y_train)
# random.best_params_

In [74]:
# random = RandomizedSearchCV(estimator=LGBMRegressor(), param_distributions=lgb_params,n_iter=100, scoring='neg_root_mean_squared_log_error', verbose=2,
#                              random_state=42,  n_jobs=-1)
# random.fit(X_train_processed, y_train)

In [75]:
# random = RandomizedSearchCV(estimator=CatBoostRegressor(), param_distributions=cat_params,n_iter=100, scoring='neg_root_mean_squared_log_error', verbose=2,
#                              random_state=42,  n_jobs=-1)
# random.fit(X_train_processed, y_train)

In [76]:
# models = {
#     "Random Forest Regressor": RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=500),
#     "Graident BoostRegressor": GradientBoostingRegressor(learning_rate=0.05, max_depth=5, min_samples_leaf=2,
#                           n_estimators=300, subsample=0.8),
#     "Xgboost Regressor": XGBRegressor(subsample= 0.7,reg_lambda= 10,reg_alpha= 0,n_estimators= 1000,min_child_weight= 1,max_depth= 4,
#                                       learning_rate= 0.03, gamma=0.1,colsample_bytree=1.0),
#     "LightGBM Regressor": LGBMRegressor(colsample_bytree=0.7, learning_rate=0.03, min_child_samples=10,
#               n_estimators=1000, num_leaves=15, reg_alpha=0.01, reg_lambda=1),
#     "CatBoost Regressor": CatBoostRegressor(bagging_temperature=1, depth=5, iterations=1000, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', random_strength=2)
   
# }

# for i in range(len(list(models))):

#     model = list(models.values())[i]

#     pipeline = Pipeline(
#         steps=[
#             ('preprocessor', preprocessor),
#             ('model', model)
#         ]
#     )

#     pipeline.fit(X_train, y_train)  # Train model

#     # Make predictions
#     y_train_pred = pipeline.predict(X_train)
#     y_test_pred = pipeline.predict(X_test)
    
#     # Evaluate Train and Test dataset
#     model_train_mae, model_train_rmse, model_train_rmsle, model_train_r2 = evaluate_model(
#         y_train, y_train_pred
#     )

#     model_test_mae, model_test_rmse, model_test_rmsle, model_test_r2 = evaluate_model(
#         y_test, y_test_pred
#     )

    
#     print(list(models.keys())[i])
    
#     print('Model performance for Training set')
#     print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
#     print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
#     print("- RMSLE: {:.4f}".format(model_train_rmsle))
#     print("- R2 Score: {:.4f}".format(model_train_r2))

#     print('----------------------------------')
    
#     print('Model performance for Test set')
#     print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
#     print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
#     print("- RMSLE: {:.4f}".format(model_test_rmsle))
#     print("- R2 Score: {:.4f}".format(model_test_r2))
    
#     print('=' * 35)
#     print('\n')

In [77]:
model=XGBRegressor(subsample= 0.7,reg_lambda= 10,reg_alpha= 0,n_estimators= 1000,min_child_weight= 1,max_depth= 4,
                   learning_rate= 0.03, gamma=0.1,colsample_bytree=1.0)
final_pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',model)
    ]
)
final_pipeline.fit(X,y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](69,)","['Id','MSSubClass','MSZoning',...,'TotalBathrooms','TotalPorchSF', 'PropertyAge']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,69
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OHE', ...), ('OE', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This sub

In [78]:
df=pd.read_csv('test.csv')

In [ ]:
df['LotFrontage']=df['LotFrontage'].fillna(df['LotFrontage'].median())
df['Alley']=df['Alley'].fillna('NA')
df['MasVnrArea']=df['MasVnrArea'].fillna(0)
df.loc[df['MasVnrArea'] == 0, 'MasVnrType'] = 'None'
df['MasVnrType'] = df['MasVnrType'].fillna(df['MasVnrType'].mode()[0])
df['BsmtQual'] = df['BsmtQual'].fillna('NA')

mask = df['BsmtQual'] == 'NA'

df.loc[mask, 'BsmtCond'] = 'NA'
df.loc[mask, 'BsmtExposure'] = 'NA'
df.loc[mask, 'BsmtFinType1'] = 'NA'
df.loc[mask, 'BsmtFinType2'] = 'NA'
df['BsmtExposure'] = df['BsmtExposure'].fillna(df['BsmtExposure'].mode()[0])
df['BsmtFinType2'] = df['BsmtFinType2'].fillna(df['BsmtFinType2'].mode()[0])
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
df.loc[df['Fireplaces']==0, 'FireplaceQu']= 'NA'
df['GarageAge'] = df['YrSold'] - df['GarageYrBlt']
df['GarageType']=df['GarageType'].fillna('NA')

mask = df['GarageType'] == 'NA'

df.loc[mask, 'GarageAge'] = 0
df.loc[mask, 'GarageFinish'] = 'NA'
df.loc[mask, 'GarageQual'] = 'NA'
df.loc[mask, 'GarageCond'] = 'NA'
# df=df.drop('GarageYrBlt',axis=1)
df['PoolQC']=df['PoolQC'].fillna('NA')
df['Fence']=df['Fence'].fillna('NA')
df['KitchenQual']=df['KitchenQual'].fillna('NA')
df['MiscFeature']=df['MiscFeature'].fillna('NA')
X=df
X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
X['Total_sqr_footage'] = X['BsmtFinSF1'] + X['BsmtFinSF2'] + X['1stFlrSF'] + X['2ndFlrSF']
X['TotalBathrooms'] = X['FullBath']+ 0.5 * X['HalfBath']+ X['BsmtFullBath']+ 0.5 * X['BsmtHalfBath']
X['TotalPorchSF'] = X['OpenPorchSF']+ X['3SsnPorch']+ X['EnclosedPorch']+ X['ScreenPorch']+ X['WoodDeckSF']
X=X.drop(['TotalBsmtSF','1stFlrSF','2ndFlrSF','BsmtFinSF1','BsmtFinSF2','FullBath','HalfBath','BsmtFullBath','BsmtHalfBath',
          'OpenPorchSF','3SsnPorch','EnclosedPorch','ScreenPorch','WoodDeckSF'],axis=1)
X['PropertyAge']=X['YrSold']-X['YearRemodAdd']
X=X.drop(['YearBuilt','YearRemodAdd',],axis=1)
X['MSSubClass']=X['MSSubClass'].astype('str')

In [83]:
features_with_na=[features for features in X.columns if X[features].isnull().sum()>=1]
for feature in features_with_na:
    print(feature,X[feature].isnull().sum(), "missing value")

In [82]:
for feature in features_with_na:
    X[feature]=X[feature].fillna(X[feature].mode()[0])

In [84]:
test_predictions = final_pipeline.predict(X)

In [85]:
submission = pd.DataFrame({
    "Id": pd.read_csv("test.csv")["Id"],
    "SalePrice": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,Id,SalePrice
0,1461,127952.718750
1,1462,168208.687500
2,1463,176988.656250
3,1464,195546.015625
4,1465,185859.343750


In [86]:
submission.shape
submission.isnull().sum()

Id           0
SalePrice    0
dtype: int64